In [1]:
%pip install -r requirements.txt
%pip install sounddevice ipywidgets


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Dependency check
What is missing and the exact commands (for THIS Python: compare with `python3 -m musikite check` in the Terminal).

In [ ]:
import sys; print(sys.executable)
from musikite.check import check
check();

# musikite — Midi object
Run from the musikite folder. `%autoreload` reloads the package after every code change.

In [2]:
%load_ext autoreload
%autoreload 2
import musikite
from musikite import Midi, Pdf, Score, Notes, StaffView, makeduet, midiplay, merge

## Soundfont (set once; new Midi objects inherit it)

In [3]:
SF2 = '/Users/utente/JIRAM_SERENA Dropbox/Alessandro Mura/personali/musica/soundfonts/FluidR3 GM.SF2'   # <- set the path

## Audio output (real time, FluidSynth)
Settings are saved in `~/.musictools/settings.json`. If nothing is heard: `audio_panel()`, change driver/device/buffer, Test.

In [4]:
Midi().add_sf2(SF2)          # soundfont for every object
musikite.audio_settings()  # current settings + output devices

{'driver': 'coreaudio',
 'device': None,
 'period_size': 64,
 'periods': 2,
 'sample_rate': 44100,
 'gain': 0.5,
 'available_devices': ['MacBook Pro Speakers',
  'Microsoft Teams Audio',
  'Soundflower (2ch)',
  'Soundflower (64ch)']}

In [5]:
musikite.audio_panel()     # driver, device, buffer, rate, gain, Save, Test

In [6]:
musikite.audio_test()      # a note + 4 clicks: active driver/device/latency, or the error

RuntimeError: pyfluidsynth missing: pip install pyfluidsynth (and brew install fluid-synth)

In [ ]:
# change from code, e.g.:
# musikite.audio_settings(device='External Headphones', period_size=128)

## Duet

In [ ]:
pippo = Midi()
pippo = makeduet(pippo, level=2, seed=42)   # same as pippo.makeduet(level=2, seed=42)
pippo.add_sf2(SF2)

makeduet(pippo, chords='Dm G7 C:Am F', rounds=3, level=2, textures='thirds,canon_bar', seed=7, bpm=90)
midiplay(pippo)

In [ ]:
musikite.audio_test()

In [ ]:
pippo.write('duetti/pippo.mid')

## From file, from PDF, merge

## Score PDF (Verovio; trumpet at written pitch)

In [ ]:
pippo.to_pdf('duetti/pippo.pdf', title='Duetto')

## Score (notation, written pitch)

In [ ]:
s = pippo.to_score('Duetto')
s

In [ ]:
s.show()

In [ ]:
s.table()

## Play in real time

In [ ]:
pippo.play()     # returns at once in Jupyter

In [ ]:
pippo.stop()

## Notes: a performance (seconds, cents, confidence)

In [ ]:
n = Notes.from_table([dict(start=0.0, dur=0.5, pitch=60.1, confidence=0.9),
                        dict(start=0.6, dur=0.5, pitch=62.3),
                        dict(start=1.2, dur=0.8, pitch=64.0)], bpm=90)
n.table()

In [ ]:
n.intonation()          # cents per written note (Bb trumpet)

## StaffView: cursor, play-along, coloured notes

In [ ]:
v = s.view()            # Qt window (layout='page' for wrapped systems)

In [ ]:
v.play(bpm=100, count_in=1, click=True, music=True)

In [ ]:
v.stop()
v.mark_errors([3, 7, 12])
v.snapshot('duetti/view.jpg')

## Main window (container of tools)

In [ ]:
app = musikite.gui()
app.set_user('Teresa')

In [ ]:
w = app.open_tool('intervals')   # Practice > Intervals
# w.start(); w.answer(7); w.next()

In [ ]:
app.audio_settings()   # Audio > Settings… dialog

## GUI tools: Generate > Duet, Convert > PDF → MIDI / MIDI → PDF score
Every object: View, Play, Stop, Save as… (.mid, .pdf, .wav) in the Objects panel.

In [ ]:
d = app.run_tool('duet', level=2, seed=42, rounds=2, chords='C F G C', textures=['thirds', 'canon_bar'])
d

In [ ]:
app.view_object(d)     # StaffView in a tab
app.play_object(d)     # real time

In [ ]:
app.stop_sound()
app.save_object(d, 'duetti/duet_gui.mid')

In [ ]:
p = app.open_file('duetti/pippo.pdf')        # any vector PDF part
app.view_object(p)

In [ ]:
m = app.run_tool('pdf2midi', p, instrument='trumpet')
app.play_object(m)

In [ ]:
app.stop_sound()
q = app.run_tool('midi2pdf', d, title='Duetto')
app.save_object(q, 'duetti/duet_gui.pdf')

In [ ]:
app.open_tool('duet')   # options panel on the right, Run

## Interval reading (staff): two notes, name the interval
Logic without GUI: `IntervalReadingQuiz` (portable); in the window: Practice > Interval reading.

In [ ]:
from musikite import IntervalReadingQuiz
from IPython.display import SVG
q = IntervalReadingQuiz(user='Teresa', answer='quality', accidentals=True, display='mixed')
q.new()
SVG(q.svg())

In [ ]:
q.answer('M3')     # e.g. 'm2' … 'P8'; with answer='number': '2' … '8'

In [ ]:
q.play()           # hear it

In [ ]:
w = app.open_tool('interval_reading')
# w.start(); w.answer('P5'); w.next()

## Exercises: blocks of N questions, points, grade 0-10, repeat
Settings > Exercises… in the window, or from code. In the window: Play (then Pause/Resume) and Stop.

In [ ]:
from musikite.exercise import Exercise, exercise_settings, set_exercise_settings
exercise_settings()

In [ ]:
# set_exercise_settings(n_questions=10, harmonic_factor=1.2, fast_s=3, slow_s=15)

In [ ]:
from musikite import IntervalQuiz
ex = Exercise(IntervalQuiz(user='Teresa'), 'intervals')
q = ex.start()      # plays the first interval
q

In [ ]:
ex.answer(q['semitones'])   # answer (here: the right one)

In [ ]:
q = ex.next(); ex.stop()    # leave: questions not answered count 0
res = ex.result
print(res.summary()); res.table()

In [ ]:
ex2 = res.repeat(IntervalQuiz(user='Teresa'))   # exactly the same questions
ex2.start()

In [ ]:
w = app.open_tool('intervals')
# w.play_clicked() / w.stop_clicked() / w.repeat()
app.exercise_settings()   # Settings > Exercises… dialog

## PDF written MuseScore style, read back by scoreread
Noteheads, rests, clefs, accidentals… are text in the embedded SMuFL font Leland: the PDF round trip gives the same notes (duet: one part per staff).

In [ ]:
d = Midi().makeduet(level=2, seed=5, key='D', chords='D G A D')
d.to_pdf('duetti/roundtrip.pdf', title='Round trip')
r = Pdf('duetti/roundtrip.pdf').to_midi()
notes = lambda m: [sorted((round(n.start, 3), n.pitch, round(n.end, 3)) for n in i.notes) for i in m.pm.instruments]
notes(d) == notes(r)

## Read & play (trumpet): microphone check
Port of MATLAB mysmartmusic: one count-in bar, metronome, recording, note-by-note check. Use loudspeakers (the mic must hear the clicks).

In [ ]:
from musikite import PlayCheck
pc = PlayCheck(d, part=0, bars=(1, 4), bpm=90)   # Midi or Score; T (trasp) = 2 for Bb trumpet
pc.score.show()

In [ ]:
r = pc.run()          # blocking: count-in, play, then evaluation
print(r['ok'], '/', r['nn'], r['rating'], 'grade', r['grade'], ' delay', r['delay'], r['delay_source'])
import pandas as pd
pd.DataFrame(r['items'])

In [ ]:
v = StaffView(pc.score).show()
v.mark_errors(pc.errors)       # wrong notes in red

In [ ]:
w = app.open_tool('play_reading')   # GUI: Practice > Read & play (trumpet)
# w.load('<object name>'); w.start()

## Exercises from a MIDI (phrases + transformations)
Every part cut into phrases (reading 8 bars, dictation 2), one transformation or 'random' (a different one, or none, per phrase).

In [ ]:
from musikite import exercises_from_midi, PlayQuiz
from musikite.exercise import Exercise
es = exercises_from_midi(d, purpose='reading', transform='random', seed=1)
es, [p.title for p in es]

In [ ]:
p = es[0]
p.to_score().show()
[p.transpose(3).title, p.reverse_melody().title, p.inversion().title, p.to_mode('dorian').title]

## Read & play as a block of exercises (same container as the other exercises)

In [ ]:
ex = Exercise(PlayQuiz(es.phrases, user='Teresa', bpm=90), 'play_reading')
q = ex.start(); q['title']

In [ ]:
pc = ex.quiz.check        # one take of the current phrase
pc.run()                  # blocking: count-in, play, record
r = ex.answer(pc.recording); r['answered'], r['rating'], r['points']
# ex.next() ... ; ex.result.summary()

## GUI: selection as input of the tools

In [ ]:
app.select('duet 1')                          # one or more objects
names = app.run_tool('exercises_reading', transform='random')   # one group per selected MIDI
app.select(names if isinstance(names, str) else names[0])
w = app.open_tool('play_reading')   # uses the selected exercises
# w.start()

## Polyphony option, grades on the exercises, click sound
polyphony=False: one exercise per part, one voice; True: all parts in one exercise (keyboard dictation). Each phrase keeps the last grade per user.

In [ ]:
es1 = exercises_from_midi(d, purpose='dictation')                   # one voice per exercise
es2 = exercises_from_midi(d, purpose='dictation', polyphony=True)   # whole duet in each exercise
es1, es2, es2[0].polyphonic

In [ ]:
es[0].grades, es[0].grade('Teresa')   # filled by Read & play in the GUI (last grade per user)

In [ ]:
from musikite.playcheck import click_track
import numpy as np
y = click_track(48000, 90, 4, 2, sf2=Midi._default_sf2)   # woodblock of the soundfont (or synthetic tock)
len(y) / 48000

## Users, folders, workspace
Every user = a separate installation (settings, results, workspace in ~/.musikite/users/<user>/). Shared folders for files: ~/musikite/<kind>.

In [ ]:
from musikite import set_current_user, folder, set_folder, save_workspace, load_workspace
set_current_user('Teresa')
[folder(k, create=False) for k in ('workspaces', 'exercises', 'midi', 'pdf')]

In [ ]:
save_workspace(folder('workspaces') + '/Teresa – test.json', {'duet': d, 'exercises': es})
objs, warnings = load_workspace(folder('workspaces') + '/Teresa – test.json')
objs, warnings

In [ ]:
# GUI: app.set_user('Paolo') saves Teresa's workspace and loads Paolo's;
# app.save_workspace_as(path), app.open_workspace(path), app.new_workspace(), app.folder_settings()

## Listen & repeat + score report
The phrase is played, then you repeat it (trumpet or MIDI keyboard); at the end a ScoreReport shows the exercise (green/red) and what you played.

In [ ]:
lr = exercises_from_midi(d, purpose='repeat')        # green exercises, 2 bars
q = PlayQuiz(lr.phrases, bpm=90, mode='repeat', source='microphone')   # or source='midi'
ex = Exercise(q, 'listen_repeat'); ph = ex.start(); ph['title']

In [ ]:
pc = ex.quiz.check; pc.run()        # phrase, count-in, your repetition (blocking)
r = ex.answer(pc.recording); r['answered'], r['rating']
score, colors = r['report'].to_score()
v = StaffView(score).show(); v.mark(colors)

In [ ]:
from musikite import audio_notes
audio_notes(pc.recording[pc.offset:], pc.fs, trasp=2)[:10]   # [start_s, dur_s, written pitch]

### Generate exercises: one tool per type, empty bars removed
Right click on a MIDI/Score > Generate exercises > Read & play / Listen & repeat (also in the Generate menu). Each type has its own options; *Remove empty bars* (default on) drops bars with rests only before cutting.

In [ ]:
from musikite import exercises_from_midi
a = exercises_from_midi(d, purpose='reading', remove_empty=True)
b = exercises_from_midi(d, purpose='reading', remove_empty=False)
len(a.phrases), len(b.phrases)
# GUI: app.run_tool('exercises_reading', remove_empty=True); app.run_tool('exercises_repeat', polyphony=True)

### Range of the exercises (Settings > Instruments)
Written pitch, default G3–F5 (C4 = middle C). Notes outside are moved by octaves into it, one by one.

In [ ]:
from musikite.phrase import instrument_range, set_instrument_range, fit_range
instrument_range('Bb (trumpet)'), instrument_range('C (piano)')
# set_instrument_range('Bb (trumpet)', 55, 77)        # G3 .. F5
es = exercises_from_midi(d, purpose='reading', note_range=(60, 72))   # override for this call
es[0].first_note(), [n for n in es[0].notes][:4]

### MIDI keyboard
Test from code (the Settings > Audio page has *Test MIDI*); Read & play with the keyboard expects the WRITTEN note.

In [ ]:
from musikite.midiin import input_names, KeyboardSynth
input_names()
# k = KeyboardSynth(program=56).open()   # play: trumpet sound; program=None = from the keyboard
# k.close()
# GUI: app.open_tool('free_player').set_instrument(0)

### Trumpet fingerings on a PDF (was scoreread.py without --midi)
Command line: `python3 -m musikite fingering part.pdf -o part.noted.pdf`

In [ ]:
from musikite import Pdf
# f = Pdf('part.pdf').fingering()          # new Pdf (temporary file); the original is not changed
# f = Pdf('part.pdf').fingering('part.noted.pdf', color='r', no_repetition=True)

### Dependency check (new computer)
`python3 -m musikite check` from the project folder: what is found, what is missing, the commands in order.

In [ ]:
import musikite
r = musikite.check()        # prints the full report; r['ok'], r['errors'], r['warnings'], r['commands']

### Theory engine (`musikite.theory`)
Pure Python (Android). Names, intervals, scales, chords, progressions, keys, chord→scale rules.

In [ ]:
from musikite import theory as th
th.note_name(70), th.note_name(70, 'it'), th.parse_note('G3'), th.interval_name(6), th.spelled_interval(5, 6)

In [ ]:
c = th.parse_chord('Dm7'); k = th.Key('C', 'major')
c, c.name('it'), sorted(c.pcs), k.roman(c), [x.name() for x in k.diatonic_chords()]

In [ ]:
bars = th.parse_progression(th.PROGRESSIONS['ii-V-i minor'])
[(t, b, ch.name(), th.RULES.scales_for(ch, key=th.Key('C', 'minor'))[0]) for t, b, ch in th.timeline(bars)]

In [ ]:
# chord -> scale with the reasons; note roles over a chord
th.RULES.scales_for('G7', next=th.parse_chord('Cm6'), explain=True)[:2], [th.RULES.classify(p, 'Cmaj7') for p in range(60, 72)]

In [ ]:
# new knowledge = a new rule (with its source)
th.RULES.add(th.Rule('blues dominant', ['7'], [('blues', 1), ('mixolydian', .9)], when=lambda c, ctx: ctx.get('key') is not None and ctx['key'].degree_of(c.root) in (0, 3), priority=5, source='example'))
r = th.RULES.scales_for('C7', key=th.Key('C')); th.RULES.remove('blues dominant'); r

### Play vs Start
Play (Objects panel) = listen: staff + red cursor + sound. Start (exercises) = practise: click (optional) + cursor, no sound.

In [ ]:
# app.select('duet 1'); sv = app.play_object()      # staff in the centre, cursor and sound
# app.stop_sound()

### Recording from the MIDI keyboard (Free player > Rec)
Every take is a part with its instrument; Save → Score.

In [ ]:
from musikite.recorder import Recorder
r = Recorder(bpm=100, ts=(4, 4))
r.start(program=56, t_zero=0)              # trumpet; times below in seconds from the first beat
for i, p in enumerate([60, 62, 64, 65]):
    r.note_on(p, 90, i * 0.6); r.note_off(p, i * 0.6 + 0.5)
r.stop()
r.to_score()

In [ ]:
# GUI: w = app.open_tool('free_player'); w.set_instrument(56); w.rec(); ...; w.rec(); w.save()

### Piano roll: edit and quantize (model without Qt)
A note is `[start_q, dur_q, pitch, velocity]`; every edit can be undone. `quantize` works as in Ableton: grid (T = triplets, `+` = mixed), strength, note ends.

In [ ]:
from musikite.midi import Midi
from musikite.pianoroll import Roll, GRIDS

m = Midi(); m.makeduet(seed=7, bpm=100)
r = m.roll()                                  # = Roll.from_midi(m)
print(r, GRIDS)
n = r.add(0, start=4, dur=1, pitch=67)        # track 0, bar 2 beat 1, a quarter
r.move(0, [n], dq=0.5, dp=2)                  # half a beat later, a tone up
r.quantize('1/8+1/8T', strength=0.8, ends=False, track=0)
r.undo(); r.redo()
r.apply_to(m)                                 # the Midi object gets the edited notes
m

In [ ]:
# a recording (Recorder parts, quarters) as a roll, quantized
from musikite.recorder import Recorder
rec = Recorder(bpm=90)
rec.parts = [dict(program=56, name='Trumpet',
                  notes=[[0.07, 0.9, 60, 90], [1.04, 0.4, 62, 90], [1.62, 0.3, 64, 90]])]
r = Roll.from_parts(rec.parts, rec.bpm, rec.ts)
r.quantize('1/8+1/8T'); print(r.tracks[0]['notes'])
rec.parts = r.to_parts()                       # what the next take plays back
r.to_midi(title='Recording')

In [ ]:
# quantize a Midi object in place
m.quantize('1/16', strength=1.0, ends=True)

In [ ]:
# piano roll window (Qt) editing m in place: hand, pencil, eraser, Quantize…, undo
# Midi.add_sf2('…/font.sf2')   # to hear the notes and use Play
v = m.pianoroll()
# v.set_position(8); v.listen(); v.stop(); v.quantize_dialog()
# GUI: a Midi object opens on the staff; button 'Piano roll' in its tab switches to this view.
# Free player: at Stop the recording is shown here and Quantize… opens; Save -> Midi object.

### MIDI keyboard: always close the port
A python-rtmidi port left open and freed by the garbage collector crashes Python at the next key (python-rtmidi 1.5.8 never destroys its C++ object). `KeyboardSynth.close()` closes it completely; a KeyboardSynth that is dropped closes it by itself. Diagnosis script: `python debug_midi.py J` (crash) / `K` (no crash).

In [ ]:
from musikite.midiin import KeyboardSynth, input_names
print(input_names())
# k = KeyboardSynth(program=0).open()     # play the keyboard with the soundfont
# k.close()                               # cancel_callback + close_port + delete

### Exercises stay exercises
A group or ONE exercise is saved only as `.json` (`musikite-exercises`); one exercise is read back as an exercise, not as a group. No MIDI / PDF / audio export.

In [ ]:
from musikite.midi import Midi
from musikite.phrase import exercises_from_midi, save_exercises, load_exercises
m = Midi(); m.makeduet(seed=3)
group = exercises_from_midi(m, purpose='reading')
one = group.phrases[0]
p = save_exercises(one, '/tmp/one_exercise')      # -> /tmp/one_exercise.json
load_exercises(p)                                  # Phrase (one exercise), not a group

### Evaluation: only the note's own pitch
Keyboard: expected and played notes paired one to one (same pitch, onsets within 0.5 quarter). Microphone: the window may move 1/4 beat, but only the note's own pitch counts (no more 'previous / next note').

In [ ]:
from musikite.transcribe import check_notes
ev = [dict(start=0, dur=1, pitch=76), dict(start=1, dur=1, pitch=67), dict(start=2, dur=1, pitch=77)]
played = [[0.0, 1, 76], [1.0, 1, 76], [2.1, 1, 77]]     # E5 played again instead of G4
[r['ok'] for r in check_notes(ev, played)[2]]            # [True, False, True]

In [ ]:
# GUI: Zoom + / Zoom - in Read & play and Listen & repeat (kept for the next exercise / report);
# with 2+ tabs a last tab 'Close all' closes them all:  app.close_all()

### Practice diary
Grade of the day = best attempt of each exercise; days frozen at 04:00; study time = twice the sound time (intervals: answer time, max 15 s per question; Free player: time really played).

In [ ]:
from musikite import diary
d = diary.get()                       # diary of the current user (~/.musikite/users/<user>/diary.json)
d.day()                               # today: exercises (best + attempts), mean per type, minutes
d.days()[-7:]                         # last days with practice

In [ ]:
d.calendar('time')                   # {day: minutes}
d.calendar('grade', kind='intervals') # {day: mean grade of the intervals}
d.candles('play_reading')[-5:]        # (day, open, high, low, close, mean)
d.trend()[-5:]                        # (day, mean, low, high, mean over 7 days)
d.weekly(), d.streak()

In [ ]:
# GUI: Analyse > Practice diary  (Chart: Calendar / Candles / Trend / Weekly time; Colour for the calendar)
# app.open_tool('practice_diary').set_chart('candles')
# Repeat of an interval block: the same questions in another order (ExerciseResult.repeat)

### Drills (Practice > Long tones, Scales & arpeggios, Lip slurs, Rhythm reading)
Every Start = ONE new exercise from the settings; Again = the same. Pure-Python logic in `drills.py`.

In [ ]:
from musikite import drills as dr
d = dr.make('scales', dict(keys=['F', 'G'], types=['major', 'harmonic minor'], octaves=1))
d.title, [n[2] + d.phrase.transposition for n in d.phrase.notes]     # written pitches
# d.phrase.to_score().show()

In [ ]:
dr.make('long_tones', dict(notes=3, beats=4)).title, \
dr.make('lip_slurs', dict(positions=['0', '2', '1'], level=3)).title, \
dr.make('rhythm', dict(level=6, bars=2, time='4/4')).info['durations']

In [ ]:
# one take (needs microphone + speakers): metronome 'high' = pure tones filtered out of the claps
# pc = d.check(bpm=72, click_mode='normal'); pc.run(); r = d.evaluate(pc); r['grade'], r['text']
# d.record(r, pc)             # practice diary
dr.lcs_match([60, 62, 64, 65], [60, 64, 65])     # scales / slurs: only the order counts -> {0, 2, 3}

In [ ]:
# transposition: a part for trumpet in D read on the Bb trumpet
from musikite.midi import Midi
from musikite.phrase import exercises_from_midi
m = Midi(); m.makeduet(seed=4)
s = exercises_from_midi(m, instrument='Trumpet in D (transposition)', level=2)
s.phrases[0].title, s.phrases[0].read_as

### Drill results in Objects; lenient clap detection
Every drill exercise is an ExerciseResult ★ (Again = another attempt, best grade). Claps: the take is analysed again at higher sensitivities, kept only if no clap appears far from the written notes.

In [ ]:
from musikite import drills as dr
d = dr.make('rhythm', dict(level=3, bars=1))
res = d.new_result(user=None)
fake = dict(grade=7.0, rating='good', text='6/8 on time', colors={0: 'green'}, fraction=0.7)
dr.Drill.add_attempt(res, fake); dr.Drill.add_attempt(res, dict(fake, grade=8.5, rating='very good'))
res.title, res.data['grade'], len(res.data['items']), dr.CLAP_SENSITIVITY

## Exercise objects with results, quick exercises, routines (2026-11-07)

No result object: an exercise done is an exercise with results (`tasks.py`). Objects has two lists: Material (top) and Exercises (bottom); exercises to do today are bold on grey.

In [ ]:
from musikite import tasks
from musikite.phrase import exercise_from_duet
ph = exercise_from_duet('reading', bars=4, level=3, part='random', seed=1)   # quick Read & play material
print(ph.title, ph.origin)
tasks.add_result(ph, 'Teresa', 7.5)
print('to do:', tasks.to_do(ph, 'Teresa'), 'grade shown:', tasks.shown_grade(ph, 'Teresa'))

In [ ]:
# exercises to do (Generate > Exercises to do)
sc = tasks.new_drill_tasks('scales', 2, keys=['F', 'G'], types=['major'])
iv = tasks.new_quiz_tasks('intervals', 1, n_questions=5, mode='harmonic')
print(sc, iv, len(iv[0].questions))

In [ ]:
# a routine: copies, 'new each time' for drills / interval blocks / duet pieces
r = tasks.Routine.from_objects(sc + iv + [ph], 'Morning', fresh=True)
print(r, [it['fresh'] for it in r.items], r.next_to_do('Teresa'))
r.prepare(0, 'Teresa'); print(r.items[0]['obj'].title)
path = tasks.save(r, '/tmp/morning'); print(tasks.load(path))

In [ ]:
# GUI
app = musikite.gui()
app.run_tool('todo_long_tones', count=2)           # Generate > Exercises to do > Long tones
w = app.open_tool('play_reading')                    # Practice > Read & play: Start = a piece of a new duet
# app.select(...); app.run_tool('routine', title='Morning', fresh=True); app.open_object('Morning')

## Changeable exercises, difficulty, canon, flash cards, chord qualities (2026-11-08)

Exercises cut from a MIDI remember it: they can be cut again (other bars, part, difficulty, instrument, transformation, tempo) — a changed exercise is a NEW exercise.

In [ ]:
from musikite.midi import Midi
from musikite import phrase as P
d = Midi().makeduet(seed=4)
st = P.exercises_from_midi(d, bars=4, level=3, practice_bpm=80)
ph = st.phrases[1]
print(ph.title, P.settings_of(ph))
new = P.remake(ph, first=3, last=6, part=1, transform='transpose +2', bpm=100)
print(new.title, new.practice_bpm, new.uid != ph.uid)

In [ ]:
from musikite.difficulty import difficulty, rank, level_name
print(difficulty(ph))
print([(s, o.title) for s, o in rank(st.phrases)][:3], level_name(difficulty(st)['score']))

In [ ]:
c = Midi().makecanon('C F G C', difficulty=3, cycles=4, seed=1)   # = python3 -m musikite canon ...
from musikite.midi import transposed, merge
print(c, transposed(c, -2), merge(c, d))

In [ ]:
from musikite.reading import FlashcardQuiz
from musikite.eartraining import ChordQuiz
f = FlashcardQuiz(answer='fingering', accidentals=True); q = f.new(); print(q['name'], f.answer(q['key'])['correct'])
cq = ChordQuiz(qualities=('', 'm', '7'), inversions=True); q = cq.new(play=False); print(q, cq.pitches(q))
# GUI: app.open_tool('flashcards'), app.open_tool('chords'), app.run_tool('canon', seed=1)

In [ ]:
# difficulty with the key: sharps/flats of the written key signature and notes outside the key
from musikite.difficulty import difficulty, key_accidentals, out_of_key
for p in st.phrases[:3]:
    d = difficulty(p)
    print(p.title, d['score'], 'key', d['key_accidentals'], 'out of key', d['out_of_key'])

In [ ]:
# key estimation without music21 (port of tonalita.m; minor if it starts / ends on the relative minor tonic)
from musikite import theory as th
print(th.estimate_major_key([62, 64, 66, 67, 69, 71, 73]))          # 2 = D major
print(th.key_of_notes([[0, 1, 57], [1, 1, 60], [2, 1, 64], [3, 1, 57]]))   # (9, 'aeolian') = A minor
print(th.key_signature(6), th.key_signature(9, 'aeolian'))

In [ ]:
# every exercise knows its instrument; the difficulty has rules per instrument (trumpet: high register)
from musikite.phrase import Phrase
from musikite.difficulty import difficulty
high = Phrase([[0, 1, 60], [1, 1, 79]], 1, practice_bpm=60)       # concert G5 = written A5 on the Bb trumpet
print(high.instrument, difficulty(high)['instrument_rules'])
print(Phrase([[0, 1, 60], [1, 1, 79]], 1, program=0).instrument)

In [ ]:
# exercises for the trumpet from a folder of MIDI files (run from the project folder):
# !python3 scripts/makeexercises.py "/personali/musica/tromba/esercizi musikite"
from musikite.phrase import exercises_from_midi
print(exercises_from_midi(d, parts=[0], bar_range=(1, 8), transform='inversion', transpose=3).phrases[0].title)

In [ ]:
# trumpet = one voice: polyphony is ignored when the exercise is played on the trumpet
st = exercises_from_midi(d, bars=4, polyphony=True, instrument='Bb (trumpet)')
print(any(p.polyphonic for p in st.phrases), st.phrases[0].instrument)

In [ ]:
# do not include MIDI: no trace of the source MIDI in the exercises
st = exercises_from_midi(d, bars=4, include_midi=False)
print(st.phrases[0].source_midi, st.phrases[0].origin, st.phrases[0].can_remake())
# script: python3 scripts/makeexercises.py FOLDER --no-include_midi

In [ ]:
# a daily routine: new exercises made with the last settings, new each time; saved as .mkr (zipped)
from musikite import tasks
r = tasks.Routine('Daily')
for k in ('long_tones', 'scales', 'intervals', 'duet'):
    r.add_new(k)
print([(it['obj'].title, it['fresh']) for it in r.items])
print(tasks.save(r, '/tmp/daily'), tasks.load('/tmp/daily.mkr'))

In [ ]:
# generated exercises are grouped: "How many" > 1 -> one TaskGroup (▦) of exercises, saved as .mke
from musikite import tasks
g = tasks.group_or_single(tasks.new_drill_tasks('lip_slurs', 5), 'Lip slurs')
print(g, [t.title for t in g.phrases])
print(tasks.save(g, '/tmp/lips'), tasks.load('/tmp/lips.mke'))

In [ ]:
# practising closes the Tools panel; settings boxes scroll on their own, so the staff is never cut
app.open_tool('todo_lip_slurs'); print('Tools open:', app.dock_tool.isVisible())
w = app.open_tool('drill_lip_slurs'); print('Tools open while practising:', app.dock_tool.isVisible())

In [ ]:
# lip slurs: ONE module (figure + rhythm) repeated on every valve position, classic order, rests between
from musikite import drills
for sd in range(3):
    d = drills.make('lip_slurs', dict(level=3, positions=['0', '2', '1', '12'], note='half'), seed=sd)
    print(d.title, d.info['figure'], d.info['rhythm'])

In [ ]:
# drag and drop into an open routine: the same as dropping, from code (selection of Objects inserted at row 0)
rt = app.run_tool('routine', title='Morning')          # on the exercises selected in Objects
w = app.practise(app.objects[rt], rt)
w.add_objects(w._selected_exercises(), at=0)           # = drop at the top of the list
print([it['obj'].title for it in app.objects[rt].items])

In [ ]:
# trumpet exercises are always in treble clef (music21 alone chose the bass clef for low lip slurs)
from musikite import drills
d = drills.make('lip_slurs', dict(level=1, positions=['23', '13', '123'], note='half'), seed=1)
print([type(c).__name__ for c in d.phrase.to_score().m21.recurse().getElementsByClass('Clef')])

In [ ]:
# Play over MIDI: up to 3 tracks shown and muted, the others played; grade n/a
from musikite.playover import PlayOver
from musikite import theory
po = PlayOver.from_midi(Midi(path))          # path: a MIDI file
print(po.readable())
po.set_tracks([0])                            # the tracks to read (0-based)
po.score()                                    # written for their instruments
print(theory.written_part(65))                # alto sax: in Eb, +9, treble clef

In [ ]:
# merge MIDI files keeping the tempo map; different tempos -> error with strict=True (scripts/merge_midi.py)
from musikite.midi import merge
m = merge('part1.from_pdf.mid', 'part2.from_pdf.mid', strict=True)
print(m.tempo_map())            # [(quarter, bpm)]: the first tempo and every change
m.write('song - all parts.mid')

In [ ]:
# progressions in Roman numerals, in any key (musikite.progressions)
from musikite.progressions import progression, roman_chord, names
print(names(style='jazz'))
print(progression('jazz blues', 'F'))
print(progression('ii-V-i minor', 'A'), '|', roman_chord('V7/ii', 'Bb'))

In [ ]:
# Chord arpeggios (drill): the chords of a progression, one sounding at every first click
from musikite import drills as dr
d = dr.make('chord_arpeggios', dict(progression='ii-V-i minor', key='D', instrument='C (piano)',
                                    clicks=8, times=1, help=True, bpm=80))
print(d.title, [c for c, _, _ in d.info['chords']])
for chord, ctx in dr.chord_context([c for c, _, _ in d.info['chords']], 'D', d.info['minor']):
    print(chord.name(), dr.chord_notes(chord, ctx))     # (chord tones, tensions): pitch classes
# pc = d.check(bpm=80); pc.run(); r = d.evaluate(pc, 'microphone')   # or app.open_tool('drill_chord_arpeggios')

In [ ]:
# Chord arpeggios, Random ("change every time"): progression, key, clicks at random; the instrument stays
from musikite import drills as dr
for _ in range(3):
    d = dr.make('chord_arpeggios', dict(random=True, instrument='Bb (trumpet)'))
    print(d.title, '|', d.info['instrument'])

In [ ]:
# Fingerings of the brass (musikite.fingerings): the WRITTEN note -> valves / slide position
from musikite.fingerings import fingering, fingering_concert, has_fingerings, INSTRUMENTS
from musikite.theory import note_name
print(list(INSTRUMENTS))
print(fingering('Trumpet in Bb', 62), fingering('Trumpet in C', 62), fingering_concert('trumpet', 60))
print(' '.join(f'{note_name(p)}={fingering("Tenor trombone", p)}' for p in range(46, 66)))
print(has_fingerings('Alto saxophone in Eb'))     # saxophones: no written fingerings

In [ ]:
# Instruments = the instrument played (trumpet, piano, trombone); random exercise + positions
from musikite import phrase as P
print(P.EXERCISE_INSTRUMENTS)                       # the transpositions (read C, play D) are hidden
ph = P.exercise_from_duet('reading', bars=4, instrument='C (trombone)')   # an octave lower, bass clef
print(ph.title, ph.instrument)
s = ph.to_score(positions=True)                     # slide positions under the notes
print([(n.nameWithOctave, [l.text for l in n.lyrics]) for n in s.m21.recurse().notes])
# s.show()

In [ ]:
# Random exercises have no "Make a variant" panel (only exercises cut from a MIDI / score)
from musikite import phrase as P
from musikite.gui.play_reading import _is_random
ph = P.exercise_from_duet('reading', bars=4)
print(ph.title, _is_random(ph))          # True: the panel is hidden

In [ ]:
# 2026-11-09 fixes: Play over MIDI written pitch, long tones search ±150 cents, 5 s after the grade
import pretty_midi
from musikite.midi import Midi
from musikite.playover import PlayOver
from musikite import drills
from musikite.gui.routine_ui import AUTO_MS
pm = pretty_midi.PrettyMIDI(initial_tempo=100)
tr = pretty_midi.Instrument(56, name='Trumpet'); tr.notes = [pretty_midi.Note(90, 62, 0, 1)]   # D concert
pm.instruments = [tr]; m = Midi(); m.pm = pm
po = PlayOver.from_midi(m, tracks=[0])
print([n.nameWithOctave for n in po.score().m21.recurse().notes], po.score().m21.atSoundingPitch)  # E4, False
print(drills.CENTS_SEARCH, AUTO_MS)

In [ ]:
# Dependency check: every library of the program is required (only the notebook ones are optional)
from musikite import check
r = check.report(deep=False)
print(r['ok'], r['errors'])
print(sorted(check.REQUIRED.values()))

In [ ]:
# Save several objects at once (GUI): app.save_objects(names) asks ONE folder; Save as on a
# multiple selection does the same. Fingerings saved next to the original:
# app.run_tool('fingering', '<pdf object>', save_next=True)   # writes <name>.noted.pdf

In [ ]:
# Grade shown in Objects and in the routine runner: only the best of TODAY (None if not done today)
from musikite import tasks
# tasks.shown_grade(obj, user) == tasks.best_today(obj, user); tasks.last_grade(obj, user) still gives the last one

In [ ]:
# Duet / canon: Key list, chords moved into the key, New = another progression (2026-11-13)
from musikite.progressions import KEYS, key_of_chords, chords_to_key, new_progression
print(KEYS)
print(key_of_chords("C F G G C F G C"))                      # C
print(chords_to_key("C F G G C F G C", "C", "E"))            # E A B B E A B E
print(chords_to_key("C F G G C F G C", "C", "Cm"))           # Cm Fm G G Cm Fm G Cm
print(new_progression("Am", current="Am Dm E Am"))           # a different one in A minor
from musikite.midi import Midi
m = Midi().makeduet(chords=chords_to_key("C F G G C F G C", "C", "E"), key="E", seed=1)

In [ ]:
# Scales with patterns / rhythms (2026-11-13), Keep, routine ◀ ▶ (GUI)
from musikite import drills as dr
d = dr.make("scales", dict(keys=["F"], types=["major"], pattern="thirds", note="varied"))
print(d.title, d.phrase.bars, d.phrase.ts, d.phrase.ts_changes)
# GUI: w = app.open_tool('scales'); w.start(); w.keep()   # into Objects at once
# Settings > Exercises: "Routine: next exercise after" (default 10 s)

In [ ]:
# Chord qualities: after a wrong answer the next question waits for the replay + 1 s (2026-11-14)
from musikite.eartraining import ChordQuiz
q = ChordQuiz(); q.new(play=False)
r = q.answer("dim7" if q.current["quality"] != "dim7" else "m")   # wrong on purpose
print(r["name"], "replay", q.feedback_s, "s -> next question after", q.feedback_s + 1.0, "s")

In [ ]:
# Routine (2026-11-14): first exercise selected at opening; ◀ / ▶ in the exercise's tab
# w = app.open_object("<routine name>")   # routine runner: Start = item 1
# w.start(); w.next(); w.previous()       # what ◀ / ▶ in the exercise's tab do

In [ ]:
# Cursor by bars (2026-11-15): Settings > Exercises "Cursor" = 2 = bars
from musikite.progress import save_setting
from musikite.staffview import cursor_mode
save_setting("cursor_mode", "bars"); print(cursor_mode())   # "notes" = the default
# sv = score.view(); sv.play()   # the cursor moves bar line to bar line
save_setting("cursor_mode", "notes")

In [ ]:
# Mac app (2026-11-17): from a Terminal on each Mac (not from Jupyter):
#   bash scripts/build_mac_app.sh      -> ~/Desktop/MusiKitE-arm64.dmg or MusiKitE-intel.dmg
import os
print(os.path.isfile("GeneralUser-GS/GeneralUser-GS.sf2"))   # soundfont put into the app

In [ ]:
# Manual inside the app + default soundfont (2026-11-17)
from musikite.gui.manual import docs_dir, slug
print(docs_dir(), slug("Read & play and Listen & repeat"))
# app.open_manual("settings.md#audio")      # Help > Manual (F1) on a section

In [ ]:
# Practice tabs show the exercise at once (2026-11-17)
# w = app.open_tool("drill_scales")   # staff of the scale made with the tab's settings
# w.new()                             # another one (= button New); w.start() does the one shown
# r = app.open_tool("play_reading"); r.preview_quick(); r.new_quick()

### Dock launcher of the source version (2026-11-17)

From Terminal, in the project folder: `bash scripts/make_launcher.sh` makes `MusiKitE (source).app` (starts `python3 -m musikite gui` from this code, MusiKitE icon); drag it to the Dock. Log of every start: `~/.musikite/launcher.log`.

In [ ]:
# Click button and Play over MIDI drag / Open… (2026-11-17)
# r = app.open_tool("play_reading"); r.click_btn.on        # audio click on?
# r.click_btn.click()                                        # = press it: click off (also during a take)
# r.sv.beat_of_bar(5.0)                                      # (beat of the bar, beats in the bar)
# po = app.open_tool("play_over")                           # nothing selected: empty tab
# po.use_file("/path/to/song.mid")                          # = drag / Open… of a file


In [ ]:
# Free player Click button; count-in always heard in Play over MIDI / Free player (2026-11-17)
# fp = app.open_tool("free_player"); fp.rec()     # count-in heard, then the clicks
# fp.click_btn.click()                            # clicks after the count-in off (live)
# fp.rec()                                        # Stop


In [ ]:
# Computer keyboard as a MIDI keyboard (2026-11-17)
from musikite import midiin
print(midiin.input_names())          # ... + 'Computer keyboard'
# k = midiin.KeyboardSynth(program=0).open(midiin.COMPUTER_KEYBOARD)
# midiin.ComputerKeyboard.key_press('a'); midiin.ComputerKeyboard.key_release('a')   # C4
# midiin.ComputerKeyboard.key_press('x')                                             # octave up
# k.close()


In [ ]:
# Note detection per instrument (2026-11-17): wind = as before, piano = by attacks
from musikite import detect
print(detect.profile('trumpet'), detect.profile('piano'), detect.profile('guitar'))
# import soundfile as sf; y, fs = sf.read('take.wav')            # a recording of yours
# for s, d, p, chord, others in detect.decaying_notes(y, fs): print(round(s, 2), p, chord, others)


In [ ]:
# Random exercises in a random key (2026-11-17): at most N sharps / flats, auto = from the level
from musikite.phrase import exercise_from_duet, random_written_key
print([random_written_key(1) for _ in range(6)])
# ph = exercise_from_duet('reading', bars=4, level=2, instrument='C (piano)')          # key: C or Am
# ph = exercise_from_duet('reading', bars=4, level=5, instrument='C (piano)', key_sig=6)


## Instruments of the user and the current instrument (2026-11-18)

The instruments played by the user (the piano always), the preferred one and the instrument studied now (the toolbar choice in the app). Exercises are made for the current instrument.

In [ ]:
from musikite import instruments as ins
print(ins.KEYS)
print(ins.user_instruments(), ins.preferred(), ins.current(), ins.played())
# ins.set_user_instruments(['Bb (trumpet)', 'Eb (alto sax)'])   # the piano is added
# ins.set_current('Eb (alto sax)')
# ins.row('Eb (alto sax)')   # transposition, GM program, clef, range, detection range

In [ ]:
# a random Read & play exercise for the current instrument
from musikite.phrase import exercise_from_duet
ph = exercise_from_duet('reading', bars=4, level=3, instrument=ins.current())
ph.to_score().show()

In [ ]:
# practice diary filtered by instrument
from musikite import diary
d = diary.get()
d.view(ins.played()).calendar('time')

## Backing track from chords (2026-11-18)

Bass, chords and drums on a progression (CONCERT chords here; the app's Generate panel writes them for your instrument). Styles: swing, bossa nova, ballad, jazz pop.

In [ ]:
from musikite import Midi
from musikite.backing import STYLES, CHORD_SOUNDS
from musikite.progressions import progression
b = Midi().makebacking('Dm7 G7 Cmaj7 Cmaj7', style='swing', bpm=140, choruses=2)
b.play()   # b.stop()

In [ ]:
# a standard: the changes of Blue Bossa in its usual key (C minor), bossa nova
chords = progression('blue bossa', 'C')
print(chords)
bb = Midi().makebacking(chords, style='bossa nova', key='Cm', sound='jazz guitar', seed=1)  # no count-in, no final chord
bb.write('blue_bossa_backing.mid')

## More ear training (2026-11-19)

Scales and modes, scale degrees, cadences by ear: blocks of questions as the intervals and the chord qualities (in the app: Practice > Scales and modes by ear / Scale degrees by ear / Cadences by ear).

In [ ]:
from musikite.eartraining import ScaleQuiz, DegreeQuiz, CadenceQuiz, SCALE_LABELS, DEGREE_LABELS, CADENCE_LABELS
q = ScaleQuiz(scales=('ionian', 'dorian', 'mixolydian', 'aeolian'), mode='up')
q.new()            # plays a scale
# q.replay()
# q.answer('dorian')

In [ ]:
d = DegreeQuiz(mode='major', degrees='diatonic')
d.new()            # cadence I IV V7 I, then a note
# d.answer(7)      # semitones above the tonic: 7 = the 5th
print(DEGREE_LABELS)

In [ ]:
c = CadenceQuiz(cadences=('perfect', 'plagal', 'half', 'deceptive'), mode='mixed')
c.new()
# c.answer('deceptive')

In [ ]:
# a block of 10 questions with grade, as the app does it
from musikite.exercise import Exercise
ex = Exercise(CadenceQuiz(mode='major'), 'cadences')
q = ex.start()
# r = ex.answer('perfect'); ex.next() …; ex.stop().summary()

## Pdf -> Score (2026-11-19)

A PDF read by scoreread and quantized to a Score (in the app: PDF → MIDI, then MIDI → Score).

In [ ]:
from musikite.pdf import Pdf
# sc = Pdf('part.pdf').to_score()
# sc.view().show()

## Euphonium (2026-11-19)

In Bb reading the treble clef (the trumpet's chart on the written note) or in C reading the bass clef.

In [ ]:
from musikite import instruments as ins
from musikite.fingerings import fingering, fingering_concert
print(ins.row('Bb (euphonium, treble clef)'), ins.row('C (euphonium)'))
print(fingering('Euphonium in Bb', 62), fingering('Euphonium', 48))   # written D4 treble = 13; C3 bass clef = 13

## Exercise library (2026-11-19)

Melodies stored as MIDI (the folder `musikite/data/library/`, built with `scripts/build_library.py MIDI_FOLDER`: transformed, crossed and generated melodies) plus the user's additions; the exercise is made when used: the key chosen (or Auto by difficulty), translated for the instrument.

In [ ]:
from musikite import library
lib = library.get()
E = lib.entries()
print(len(E), E[0] if E else 'empty library: run scripts/build_library.py MIDI_FOLDER')

In [ ]:
# one melody as an exercise for the tuba (Auto key for level 2) and for the euphonium in D
e = E[0]
tuba = lib.exercise(e, 'Bb (tuba)', level=2)
euph = lib.exercise(e, 'Bb (euphonium, treble clef)', written_tonic=2)
print(tuba.title, tuba.uid, tuba.origin['translation'])
print(lib.difficulty(e, 'Bb (tuba)'), lib.difficulty(e, 'Bb (trumpet)'))
# tuba.to_score().show()

In [ ]:
# translate any exercise
from musikite.translate import translate, min_note_s
print([round(min_note_s('Bb (sousaphone)', p), 2) for p in (34, 46, 58)])
# ph2 = translate(ph, 'C (bassoon)')

In [ ]:
# done by the current user (any attempt, any key) and adding to the user library
done = lib.done_uids()
print(sum(lib.is_done(x, 'Bb (trumpet)', done) for x in E), 'done')
# entry, added = lib.add(Midi('my_melody.mid'), 'my melody')   # or an exercise Phrase

In [ ]:
# how the library is made (libgen): melody of a MIDI, entropy, the transformations ('inversion': 7, 'all': 15)
from musikite import libgen
from musikite.midi import Midi
# src = libgen.melody_of_midi(Midi('piece.mid'))
# print(libgen.entropy(src)); [t for t, _ in libgen.transformations(src, 'inversion')]

In [ ]:
# new instruments and their fingerings
from musikite import instruments as ins
from musikite.fingerings import fingering
print([ins.label(k) for k in ins.KEYS])
print(fingering('Tuba in Bb', 36), fingering('Horn in F', 62), fingering('Tuba in Eb', 48))

## Power Tab -> MIDI (2026-11-19)

Power Tab Editor 1.7 files (.ptb); a whole folder: `scripts/ptb2midi.py IN OUT`.

In [ ]:
from musikite import powertab
# song = powertab.read('song.ptb'); [(i.name, i.program, i.strings, i.capo) for i in song.scores[0].infos]
# powertab.convert('song.ptb', 'song.mid')
# m = powertab.to_midi('song.ptb'); m.play()

## Guitar Pro -> MIDI (2026-11-19)

Guitar Pro files (.gp3, .gp4, .gp5; needs `pip install pyguitarpro`); a whole folder: `scripts/gp2midi.py IN OUT`. Both readers write only the first time signature.

In [ ]:
from musikite import gptab
# gptab.convert('song.gp3', 'song.mid', monophonic=True)
# m = gptab.to_midi('song.gp3'); m.play()

## Melody track of a MIDI (2026-11-19)

The track that most looks like a melody, written alone; a whole folder: `scripts/midi2melody.py IN OUT`.

In [ ]:
from musikite import melodytrack
# [(c['track'], c['channel'], c['name'], round(c['score'], 2)) for c in melodytrack.candidates('song.mid')]
# melodytrack.convert('song.mid', 'song.melody.mid', monophonic=True)

## Library: column Type (2026-11-20)

Original, or the transformation of a transformed melody (dorian, inversion, retrograde, lydian…).

In [ ]:
from musikite import library
lib = library.get()
[(e['title'], library.type_label(e)) for e in lib.entries()[:10]]

## Preview of an object (2026-11-20)

One click in Objects = the tab Preview (one only; it disappears when another tab comes to the front). From code:

In [ ]:
# app = musikite.gui()
# app.preview_object(app.selected_object())

## Note names follow the key signature (2026-11-20)

A# in F# major (6 sharps), F# in G minor (Bb major signature + leading tone), Bb in C major.

In [ ]:
from musikite import theory
[theory.spell_pitch(70, 6), theory.spell_pitch(66, -2), theory.spell_pitch(70, 0), theory.spell_pitch(65, 6)]

## One exercise bar, Edit on the right (2026-11-20)

Start/Stop in one button, Again (= the old Repeat too); Read & play: bpm, input, Show positions, Edit (Make a variant in the right panel, Run = the variant in Objects, selected). A new object is always selected.

In [ ]:
# app = musikite.gui()
# w = app.open_exercise(app.selected_object())   # a Read & play exercise cut from a MIDI
# w.toggle_edit(); w.e_level.setCurrentIndex(1); w.run_variant()
# app.selected_object()                         # the variant

## Duet from a melody (2026-11-20)

`accompany.duet_from_melody(src, mode=...)`: the melody of a MIDI / score / exercise (a MIDI: the track with the most notes, or `track=`) and a second voice made for it. Modes: `harmony` (thirds / sixths in the key, chord tones where the harmony asks), `canon` (the melody delayed; refused below 70 % consonance unless `allow_dissonant=True`), `accompaniment` (the chords found, played as `long notes`, `half notes`, `beats`, `arpeggio`). A NEW Midi; `duet_info` keeps what was done. GUI: Generate > Duet from a melody.

In [ ]:
from musikite.midi import Midi
from musikite import accompany
src = Midi().makeduet(seed=7)                      # any MIDI: here a duet, its track 0 is the melody
accompany.chords_of(src, track=0)                  # the chords found for that melody

In [ ]:
for mode in accompany.MODES:
    try:
        d = accompany.duet_from_melody(src, mode=mode, track=0, instrument="Bb (trumpet)")
        print(d, d.duet_info, "\n")
    except ValueError as e:                        # e.g. a melody that does not hold a canon
        print(mode, ":", e, "\n")

In [ ]:
d = src.duet_from_melody(mode="accompaniment", track=0, pattern="beats", instrument="C (trombone)")
d.to_score("accompaniment, trombone").show()
# d.add_sf2("musikite/data/GeneralUser-GS.sf2"); d.play()

In [ ]:
d = src.duet_from_melody(mode="canon", track=0, delay="1 bar", canon_interval="unison / octave",
                         allow_dissonant=True)
d.duet_info["consonance"], d.duet_info["consonance_after"], d.duet_info["moved"]

In [ ]:
# an exercise (its own notes and key):
# from musikite.phrase import exercise_from_duet
# ph = exercise_from_duet("reading", bars=8)
# accompany.duet_from_melody(ph, mode="harmony", interval="sixths").to_score().show()
# command line: python3 -m musikite second_voice melody.mid -m canon --delay "2 bars" -o duet.mid

## Play from a bar (2026-11-23)

Click a bar on a staff = the next Play starts there after one count-in bar; double click = it starts at once. From code:

In [ ]:
from musikite import Midi
from musikite.staffview import StaffView
m = Midi().makeduet(seed=1)
sv = StaffView(m).show()
sv.from_q = sv.bar_start(9.0)    # as a click on bar 3 (4/4)
sv.listen()                       # count-in bar, then from bar 3

## Play over MIDI with the positions (2026-11-23)

In [ ]:
from musikite.playover import PlayOver
po = PlayOver.from_midi(m, [0])
print(po.has_positions())
po.score(positions=True).show()   # valves under the notes

## Batch 2026-11-24

The package is now `musikite` (`import musictools` still works: the same modules).

In [ ]:
from musikite import Midi
q = Midi().makeensemble('quartet', chords='C F G C', level=2, seed=3)
q.to_score().show()

In [ ]:
from musikite.melodies import make_melodies
st = make_melodies(3, 'Bb (trumpet)', key='Auto', difficulty=None, bars=8, seed=1)
st.title, [ph.title for ph in st.phrases]

In [ ]:
from musikite import fingerings as fg
print(fg.fingering('Tuba in Bb', 33, opts=dict(four_valves=True)))   # A1 with the 4th valve
print(fg.fingering('Tenor trombone', 48, opts=dict(f_trigger=True)))  # C3 = T1
print(fg.fingering('Horn in F', 69, opts=dict(double_horn=True)))     # written A4 = T0

In [ ]:
from musikite import rework
ph = st.phrases[0]
print(len(rework.split(ph)), rework.resample(ph, 0.5).notes[:4])
q = rework.quantize([0, .26, .5, 1.01, 1.24], [60, 62, 64, 65, 67])
q.notes, q.bpm

In [ ]:
from musikite import drills as dr
d = dr.make('improvisation', dict(progression='ii-V-I major', key='C', style='bossa nova', choruses=2, bpm=130))
d.title, d.info['chords'][:3]

## Melody engines and the counterpoint duet (2026-11-24)

Melodies made by rules (`melodygen.py`: form, harmony, rhythm, then the notes by an engine: `skeleton` Povel, `probabilistic` Temperley, `degrees` Markov on scale degrees) — no library. Duets: `engine='counterpoint'` (`duetcp.py`).

In [ ]:
from musikite import melodygen
# one melody, concert pitch: [[start q, dur q, pitch]]
notes = melodygen.melody(bars=16, ts='4/4', tonic=0, mode='ionian', lo=60, hi=79, level=4, engine='skeleton', seed=1)
notes[:8]

In [ ]:
from musikite.melodies import make_melodies
for eng in ('skeleton', 'probabilistic', 'degrees', 'duet voice'):
    st = make_melodies(2, 'Bb (trumpet)', key='Auto', difficulty=4, bars=16, seed=3, engine=eng, time='4/4')
    print(st.title)
st.phrases[0].to_score().show()

In [ ]:
from musikite import Midi
m = Midi().makeduet(level=2, seed=7, engine='counterpoint')
m.to_score().show()

## Editor tools of the piano roll (2026-11-25)

Bars, Double, Resample, Quantize from playing on a `pianoroll.Roll`; an edited exercise and Split / Adapt (`rework`).

In [ ]:
from musikite import Midi, pianoroll, rework
m = Midi()
m.makeduet(seed=1)
r = pianoroll.Roll.from_midi(m)
print(r)
r.insert_bars(3, 2)            # two empty bars before bar 3
r.delete_bars(3, 4)            # and away again
r.double(keep_speed=True)      # values x 2, tempo x 2
print(r, r.bpm)
r.undo(); print(r, r.bpm)      # back
r.resample(0, 0.5)             # track 0 as a line of eighths
m2 = r.to_midi("resampled")
m2

In [ ]:
# timing from the playing of a track (a piece played without metronome)
r = pianoroll.Roll.from_midi(m)
T0, bpm = r.quantize_from_playing(0)
print(f"eighth = {T0*1000:.0f} ms, bpm = {bpm:.0f}")

In [ ]:
# an exercise edited in the piano roll -> a NEW exercise; Split / Adapt
from musikite import phrase
ph = phrase.exercise_from_duet("reading", bars=8)
mm = ph.to_midi()
new, info = phrase.edited(ph, mm, ph.length_q)
print(new, info)
print(rework.split(ph))
print(rework.adapt(ph))